# Bài 1.4 · Pipeline và data leakage

**Câu hỏi của buổi này:** điền ô trống, chuẩn hóa, mã hóa cột chữ: các bước ấy học từ đâu, và nếu học nhầm chỗ thì điểm kiểm định sai đến mức nào?

Dữ liệu chính là **Adult** (UCI, điều tra dân số Mỹ 1994): gần 49 000 người, 14 cột có cả số, chữ và ô trống; nhiệm vụ là đoán thu nhập năm có vượt 50 000 USD không.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng mẫu, biểu đồ ô thiếu theo cột |
| 2. Pipeline | sơ đồ Pipeline, bảng những gì nó học từ train, một dòng đi qua từng bước |
| 3. Nhóm mới | một quốc gia chưa từng gặp đi qua Pipeline |
| 4. Rò rỉ khi chọn feature | điểm 0.5 thành 0.9 trên nhãn ngẫu nhiên |
| 5. Rò rỉ khi mã hóa theo nhãn | một cột gần như ID cho AUC gần 1 |
| 6. Fit lại trong từng fold | bảng những gì mỗi fold học, AUC từng fold |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 1.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/1.4).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style
    if fmt:
        st = st.format(fmt)
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import sklearn
from sklearn import set_config
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score, accuracy_score

set_config(display='diagram')
table(pd.DataFrame([('scikit-learn', sklearn.__version__), ('pandas', pd.__version__), ('numpy', np.__version__)],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Cột chữ giữ nguyên dạng chữ; ô trống là `NaN`. Model không nhận trực tiếp hai thứ này, nên cần tiền xử lý.

In [ ]:
adult = fetch_openml('adult', version=2, as_frame=True)
df = adult.data.copy()
y = (adult.target == '>50K').astype(int).to_numpy()
cat_cols = df.select_dtypes(['category', 'object']).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols]

table(df.head(6), 'Sáu dòng đầu của Adult')
miss = df.isna().sum()
miss = miss[miss > 0].sort_values()
table(pd.DataFrame({'cột': miss.index, 'kiểu': ['chữ' if c in cat_cols else 'số' for c in miss.index], 'số ô thiếu': miss.values, 'tỉ lệ': miss.values / len(df)}),
      'Các cột có ô thiếu', fmt={'tỉ lệ': '{:.1%}'})
check(f'{len(df)} người, {len(num_cols)} cột số, {len(cat_cols)} cột chữ', len(df) == 48842)
check(f'tỉ lệ thu nhập >50K là {y.mean():.3f}', 0.2 < y.mean() < 0.3)

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.barh(miss.index, miss.values, color=C['clay']); ax.set_xlabel('số ô thiếu')
ax.set_title('Ô thiếu nằm ở ba cột chữ'); ax.grid(axis='y', visible=False); bar_labels(ax, '{:.0f}')
plt.tight_layout(); plt.show()

## 2. Pipeline và ColumnTransformer

Mỗi loại cột có một chuỗi bước riêng:

- **cột số**: điền ô thiếu bằng median, rồi chuẩn hóa $z=(x-\mu)/\sigma$;
- **cột chữ**: điền ô thiếu bằng giá trị gặp nhiều nhất, rồi one-hot.

Median, $\mu$, $\sigma$, danh sách nhóm đều là **những con số được học**. Đặt mọi bước vào một `Pipeline` thì `fit` chỉ chạm vào train; `predict` dùng lại đúng những con số đó cho dữ liệu mới. Sơ đồ dưới bấm vào được.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, stratify=y, random_state=0)
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_cols)])
model = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=3000))])
model.fit(X_tr, y_tr)
display(model)

In [ ]:
num_pipe = model.named_steps['pre'].named_transformers_['num']
cat_pipe = model.named_steps['pre'].named_transformers_['cat']
table(pd.DataFrame({'cột số': num_cols, 'median để điền': num_pipe[0].statistics_, 'μ': num_pipe[1].mean_, 'σ': num_pipe[1].scale_}),
      f'Pipeline học từ {len(X_tr)} dòng train: cột số', fmt={'median để điền': '{:,.0f}', 'μ': '{:,.1f}', 'σ': '{:,.1f}'})
table(pd.DataFrame({'cột chữ': cat_cols, 'giá trị để điền': cat_pipe[0].statistics_, 'số nhóm đã học': [len(c) for c in cat_pipe[1].categories_]}),
      'Pipeline học từ train: cột chữ')

p_te = model.predict_proba(X_te)[:, 1]
acc, auc = accuracy_score(y_te, p_te >= 0.5), roc_auc_score(y_te, p_te)
table(pd.DataFrame([('accuracy', acc), ('ROC-AUC', auc)], columns=['metric', 'test']), f'Chấm trên {len(X_te)} dòng test', fmt={'test': '{:.4f}'})
check('ROC-AUC trên test vượt 0.85', auc > 0.85)

Một dòng test đi qua Pipeline: cột số thành một số $z$, cột chữ thành đúng một cột one-hot bằng 1.

In [ ]:
row = X_te[X_te['native-country'].isna()].iloc[[0]]      # chọn một dòng có ô trống để thấy bước điền
fill = dict(zip(cat_cols, cat_pipe[0].statistics_))
pre_fit = model.named_steps['pre']
z = pre_fit.transform(row)
z = z.toarray()[0] if hasattr(z, 'toarray') else z[0]
names = list(pre_fit.get_feature_names_out())
trace = []
for c in num_cols:
    trace.append((c, row[c].iloc[0], f"z = {z[names.index('num__' + c)]:+.2f}"))
for c in cat_cols:
    on = [n for n in names if n.startswith(f'cat__{c}_') and z[names.index(n)] == 1]
    raw = row[c].iloc[0]
    trace.append((c, raw if pd.notna(raw) else f'trống → điền {fill[c]}', on[0].replace('cat__', '') + ' = 1'))
table(pd.DataFrame(trace, columns=['cột', 'giá trị gốc', 'sau Pipeline']), 'Một dòng test trước và sau tiền xử lý')
check(f'vector sau Pipeline dài {len(z)}, có {int((z != 0).sum())} số khác 0', len(z) == len(names))
y_row = y_te[X_te.index.get_loc(row.index[0])]
note(f'Ô native-country trống được điền <b>{fill["native-country"]}</b>, giá trị gặp nhiều nhất trong train. '
     f'Xác suất thu nhập >50K của người này: <b>{model.predict_proba(row)[0, 1]:.3f}</b>; nhãn thật: {">50K" if y_row else "≤50K"}.')

## 3. Một nhóm chưa từng gặp

Ở test, có thể xuất hiện một quốc gia không có trong train. `handle_unknown='ignore'` cho mọi cột one-hot của quốc gia bằng 0 thay vì báo lỗi.

In [ ]:
new = row.astype(object).copy()
new['native-country'] = 'Atlantis'
zn = pre_fit.transform(new)
zn = zn.toarray()[0] if hasattr(zn, 'toarray') else zn[0]
country_cols = [i for i, n in enumerate(names) if n.startswith('cat__native-country_')]
table(pd.DataFrame([(f'trống → {fill["native-country"]}', int(z[country_cols].sum()), model.predict_proba(row)[0, 1]),
                    ('Atlantis', int(zn[country_cols].sum()), model.predict_proba(new)[0, 1])],
                   columns=['quốc gia', f'số cột quốc gia bằng 1 (trong {len(country_cols)})', 'xác suất >50K']),
      'Cùng một người, đổi quốc gia sang một giá trị chưa từng gặp', fmt={'xác suất >50K': '{:.3f}'})
check('quốc gia lạ không bật cột one-hot nào', zn[country_cols].sum() == 0)

## 4. Rò rỉ khi chọn feature

Thí nghiệm kinh điển: 200 mẫu, 5 000 feature **ngẫu nhiên**, nhãn cũng **tung đồng xu**. Không model nào đoán đúng hơn 50% được, vì chẳng có gì để học.

- **Cách sai**: chọn 50 feature tương quan nhất với nhãn **trên cả bảng**, rồi mới cross-validation.
- **Cách đúng**: đặt bước chọn feature vào Pipeline, để mỗi fold tự chọn trên train của nó.

Lặp lại với 20 bộ dữ liệu ngẫu nhiên khác nhau.

In [ ]:
def leak_experiment(seed):
    rng = np.random.default_rng(seed)
    Xn = rng.normal(size=(200, 5000)); yn = rng.integers(0, 2, 200)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    Xsel = SelectKBest(f_classif, k=50).fit(Xn, yn).transform(Xn)
    wrong = cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=cv).mean()
    right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=50), LogisticRegression(max_iter=1000)), Xn, yn, cv=cv).mean()
    return wrong, right

runs = pd.DataFrame([leak_experiment(s) for s in range(20)], columns=['chọn trên cả bảng (sai)', 'chọn trong từng fold (đúng)'])
fig, ax = plt.subplots(figsize=(8, 3.6))
for k, (col, c) in enumerate(zip(runs.columns, [C['clay'], C['blue']])):
    ax.scatter(runs[col], np.full(len(runs), k) + np.random.default_rng(k).uniform(-.12, .12, len(runs)), color=c, s=36, alpha=.8)
    ax.text(runs[col].mean(), k + .3, f'trung bình {runs[col].mean():.2f}', ha='center', fontsize=10, color=c)
ax.axvline(0.5, color=C['gray'], ls='--'); ax.text(0.505, -0.45, 'đoán bừa', color=C['gray'], fontsize=9)
ax.set_yticks([0, 1], runs.columns); ax.set_ylim(-.6, 1.6); ax.set_xlim(0.2, 1.0)
ax.set_xlabel('accuracy cross-validation (20 bộ dữ liệu, mỗi chấm một bộ)'); ax.set_title('Nhãn ngẫu nhiên: chỉ cách sai mới thấy "học được"')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
table(runs.agg(['mean', 'std']).T.reset_index(names='cách làm'), 'Accuracy trên dữ liệu không có gì để học', fmt={'mean': '{:.3f}', 'std': '{:.3f}'})
check('cách đúng quanh 0.5', abs(runs.iloc[:, 1].mean() - 0.5) < 0.05)
note(f"Bước chọn feature đã nhìn nhãn của cả fold validation. Điểm ảo <b>{runs.iloc[:, 0].mean():.2f}</b> chỉ là trí nhớ về chính những nhãn ấy; cách đúng ra {runs.iloc[:, 1].mean():.2f}.")

## 5. Rò rỉ khi mã hóa theo nhãn

**Mã hóa theo nhãn** (target encoding) thay mỗi nhóm bằng tỉ lệ nhãn 1 của nhóm đó. Cột `fnlwgt` là trọng số mẫu của cuộc điều tra: hơn 28 000 giá trị khác nhau cho gần 49 000 người. Coi nó là cột nhóm thì phần lớn nhóm chỉ có một hai người.

- **Cách sai**: tính tỉ lệ trên cả bảng, rồi cross-validation.
- **Cách đúng**: trong mỗi fold, tính tỉ lệ chỉ trên train của fold; nhóm không có trong train nhận tỉ lệ chung.

In [ ]:
g = df['fnlwgt'].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
full_map = pd.Series(y).groupby(g).mean()
enc_full = pd.Series(g).map(full_map).to_numpy()

table(pd.DataFrame({'fnlwgt': g[:6], 'nhãn thật': y[:6], 'mã hóa trên cả bảng': enc_full[:6]}), 'Sáu dòng đầu: giá trị mã hóa gần như chép lại nhãn', fmt={'mã hóa trên cả bảng': '{:.2f}'})

auc_wrong, auc_right = [], []
for a, b in cv.split(g, y):
    auc_wrong.append(roc_auc_score(y[b], enc_full[b]))
    fold_map = pd.Series(y[a]).groupby(g[a]).mean()
    enc_b = pd.Series(g[b]).map(fold_map).fillna(y[a].mean()).to_numpy()
    auc_right.append(roc_auc_score(y[b], enc_b))
res5 = pd.DataFrame({'fold': range(1, 6), 'mã hóa trên cả bảng (sai)': auc_wrong, 'mã hóa trong từng fold (đúng)': auc_right})

ax = res5.set_index('fold').plot.bar(figsize=(9, 3.6), color=[C['clay'], C['blue']], width=.75, rot=0)
ax.axhline(0.5, color=C['gray'], ls='--'); ax.set_ylim(0, 1.12); ax.set_ylabel('ROC-AUC trên fold validation')
ax.set_title('Một cột gần như ID: chỉ cách sai mới thấy AUC cao'); ax.legend(loc='upper left', bbox_to_anchor=(1, 1)); bar_labels(ax, '{:.2f}')
plt.tight_layout(); plt.show()
check(f'{len(full_map):,} giá trị fnlwgt khác nhau trên {len(g):,} người', len(full_map) > 20000)
note(f'Cách sai cho AUC trung bình <b>{np.mean(auc_wrong):.3f}</b> vì mã hóa đã nhìn nhãn của chính dòng đang chấm; cách đúng còn {np.mean(auc_right):.3f}.')

## 6. Pipeline được fit lại trong từng fold

Đưa cả Pipeline vào `cross_validate`: mỗi fold tạo một bản sao mới, fit median, $\mu$, $\sigma$ và danh sách nhóm trên đúng 4/5 dữ liệu train của fold đó.

In [ ]:
cvres = cross_validate(model, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc', return_estimator=True)
i_age, i_hours = num_cols.index('age'), num_cols.index('hours-per-week')
folds = pd.DataFrame([(k + 1, est.named_steps['pre'].named_transformers_['num'][1].mean_[i_age],
                       est.named_steps['pre'].named_transformers_['num'][1].mean_[i_hours], s)
                      for k, (est, s) in enumerate(zip(cvres['estimator'], cvres['test_score']))],
                     columns=['fold', 'μ của age', 'μ của hours-per-week', 'ROC-AUC'])
table(folds, 'Mỗi fold học μ riêng trên train của nó', fmt={'μ của age': '{:.3f}', 'μ của hours-per-week': '{:.3f}', 'ROC-AUC': '{:.4f}'})

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(folds['fold'].astype(str), folds['ROC-AUC'], color=C['blue']); bar_labels(ax, '{:.4f}')
ax.set_ylim(folds['ROC-AUC'].min() - 0.01, folds['ROC-AUC'].max() + 0.006); ax.set_xlabel('fold'); ax.set_title('ROC-AUC từng fold')
plt.tight_layout(); plt.show()
check('5 bản Pipeline khác nhau, μ của age không trùng nhau', folds['μ của age'].nunique() == 5)
note(f"AUC 5 fold: trung bình <b>{folds['ROC-AUC'].mean():.4f}</b>, lệch chuẩn {folds['ROC-AUC'].std():.4f}. Điểm test ở mục 2 là {auc:.4f}.")

## 7. Tóm tắt

In [ ]:
summary = [('ROC-AUC trên test, Pipeline đúng', f'{auc:.4f}'),
           ('nhãn ngẫu nhiên, chọn feature trên cả bảng', f'{runs.iloc[:, 0].mean():.3f} accuracy'),
           ('nhãn ngẫu nhiên, chọn feature trong fold', f'{runs.iloc[:, 1].mean():.3f} accuracy'),
           ('fnlwgt mã hóa theo nhãn trên cả bảng', f'{np.mean(auc_wrong):.3f} AUC'),
           ('fnlwgt mã hóa theo nhãn trong fold', f'{np.mean(auc_right):.3f} AUC')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 4, đổi `k=50` thành 5 hay 500 rồi xem điểm ảo thay đổi; ở mục 5, thay `fnlwgt` bằng `occupation` (ít nhóm, mỗi nhóm đông người) để thấy rò rỉ nhỏ hẳn; thử bỏ `handle_unknown='ignore'` ở mục 3.